# Bài 5
Đây là notebook chứa mã nguồn đầy đủ của bài 5.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
PROJECTS = [
    # id, name, cost, npv, cost_y12, cost_y35, gdp_imp, equity, ai_ready
    (1, 'TT Dữ liệu Hòa Lạc', 12000, 21500, 8500, 3500, 15, 2, 8),
    (2, 'TT Dữ liệu phía Nam', 11500, 20800, 7500, 4000, 14, 3, 7),
    (3, '5G toàn quốc', 18000, 32500, 12000, 6000, 20, 10, 5),
    (4, 'VNeID 2.0', 4500, 9200, 3500, 1000, 5, 8, 2),
    (5, 'Dịch vụ công v3', 3200, 6800, 2500, 700, 4, 9, 2),
    (6, 'Y tế số', 5800, 11400, 4000, 1800, 6, 12, 4),
    (7, 'Giáo dục số', 6500, 12200, 4500, 2000, 5, 15, 5),
    (8, 'TT AI quốc gia', 15000, 28500, 9000, 6000, 18, 2, 25),
    (9, 'Sandbox Fintech', 2500, 5800, 1800, 700, 6, 4, 3),
    (10, 'Logistics số', 7200, 13800, 5000, 2200, 10, 3, 6),
    (11, 'Nông nghiệp số', 4800, 8500, 3500, 1300, 5, 14, 3),
    (12, 'Đào tạo 50k kỹ sư', 8500, 16200, 5500, 3000, 8, 10, 15),
    (13, 'KCN Bán dẫn', 20000, 35000, 13000, 7000, 25, 2, 10),
    (14, 'An ninh mạng SOC', 3800, 7500, 2800, 1000, 3, 3, 8),
    (15, 'Open Data', 1500, 3800, 1200, 300, 2, 8, 4),
]

In [ ]:
def solve_bai05(budget=80000, w_gdp=0.40, w_equity=0.30, w_ai=0.30):
    P = list(range(1, 16))
    
    C = {p[0]: p[2] for p in PROJECTS}
    B = {p[0]: p[3] for p in PROJECTS}
    C12 = {p[0]: p[4] for p in PROJECTS}
    gdp_imp = {p[0]: p[6] for p in PROJECTS}
    equity = {p[0]: p[7] for p in PROJECTS}
    ai_ready = {p[0]: p[8] for p in PROJECTS}
    proj_names = {p[0]: p[1] for p in PROJECTS}

    # Xác suất hoàn thành đúng tiến độ
    # Hạ tầng: P1,P2,P3 = 0.85; Chính phủ số: P4,P5 = 0.75; AI/bán dẫn: P8,P13 = 0.65; còn lại: 0.80
    prob_completion = {}
    for p in PROJECTS:
        pid = p[0]
        if pid in [1, 2, 3]:       # hạ tầng
            prob_completion[pid] = 0.85
        elif pid in [4, 5]:        # chính phủ số
            prob_completion[pid] = 0.75
        elif pid in [8, 13]:       # AI/bán dẫn
            prob_completion[pid] = 0.65
        else:
            prob_completion[pid] = 0.80

    def _solve_mip(bud, force_p1p2=False, use_risk=False):
        m = pulp.LpProblem('VN_MIP', pulp.LpMaximize)
        y = pulp.LpVariable.dicts('y', P, cat='Binary')
        
        obj = []
        for i in P:
            base_b = B[i] * (1.0 + w_gdp*gdp_imp[i]/20.0 + w_equity*equity[i]/10.0 + w_ai*ai_ready[i]/15.0)
            if use_risk:
                base_b = prob_completion[i] * B[i]
            obj.append(base_b * y[i])
        m += pulp.lpSum(obj)
        
        m += pulp.lpSum(C[i]*y[i] for i in P) <= bud
        m += pulp.lpSum(C12[i]*y[i] for i in P) <= bud / 2.0
        
        if force_p1p2:
            m += y[1] >= 1  # Bắt buộc P1
            m += y[2] >= 1  # Bắt buộc P2
        else:
            m += y[1] + y[2] <= 1  # C3: Loại trừ
        
        m += y[8] <= y[12]   # C4
        m += y[13] <= y[12]  # C5
        m += y[4] + y[5] >= 1  # C6
        m += y[14] >= 1        # C6
        m += pulp.lpSum(y[i] for i in P) >= 7
        m += pulp.lpSum(y[i] for i in P) <= 11
        
        m.solve(pulp.PULP_CBC_CMD(msg=False))
        ok = m.status == pulp.LpStatusOptimal
        
        if ok:
            sel_ids = [i for i in P if pulp.value(y[i]) > 0.5]
            sel_names = [proj_names[i] for i in sel_ids]
            total_cost = sum(C[i] for i in sel_ids)
            total_npv = sum(B[i] for i in sel_ids)
            total_val = pulp.value(m.objective)
        else:
            sel_ids, sel_names = [], []
            total_cost, total_npv, total_val = 0, 0, 0

        return {
            'status': 'Optimal' if ok else 'Infeasible',
            'Z': total_val,
            'cost': total_cost,
            'total_npv': total_npv,
            'npv_margin': round(total_val / (total_cost + 1e-9), 4),
            'selected': sel_names,
            'selected_ids': sel_ids,
        }

    # === (1) Giải gốc ===
    res_base = _solve_mip(budget)
    
    # Chi tiết dự án được chọn
    project_details = {}
    for p in PROJECTS:
        if p[1] in res_base['selected']:
            project_details[p[1]] = {
                'cost': C[p[0]], 'npv': B[p[0]],
                'gdp_impact': gdp_imp[p[0]], 'equity': equity[p[0]], 'ai_readiness': ai_ready[p[0]],
            }

    # === (2) Nới ngân sách lên 100.000 tỷ ===
    res_100k = _solve_mip(100000)

    # === (3) Bắt buộc P1 + P2 (redundancy) ===
    res_p1p2 = _solve_mip(budget, force_p1p2=True)

    # === (4) Tối đa E[Z] với rủi ro ===
    res_risk = _solve_mip(budget, use_risk=True)

    return {
        'status': res_base['status'],
        'Z': res_base['Z'],
        'cost': res_base['cost'],
        'total_npv': res_base['total_npv'],
        'npv_margin': res_base['npv_margin'],
        'selected': res_base['selected'],
        'projects': project_details,
        'prob_completion': {proj_names[k]: v for k, v in prob_completion.items()},
        # Scenarios
        'res_100k': res_100k,
        'res_p1p2': res_p1p2,
        'res_risk': res_risk,
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai05()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())